# IDF sensitivity (README section 4.1)

A corpus edit changes `df` by at most one per term, but `idf = ln(N/df)` is
steeply non-linear at small `df`, so the induced shift is far from uniform. A
rare term moves orders of magnitude more than a common one.

In [ ]:
import sys
from pathlib import Path

# Run from anywhere: notebooks/ is a sibling of src/.
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))

In [ ]:
from tfidf_stability.datasets.synthetic import SyntheticSpec, generate
from tfidf_stability.perturbation.corpus_edits import add_document
from tfidf_stability.perturbation.idf_perturb import analyse_idf_shift
from tfidf_stability.preprocessing.pipeline import PreprocessingPipeline
from tfidf_stability.vectorisation.tfidf import TfidfVectoriser

corpus_data = generate(SyntheticSpec(n_docs=300, vocab_size=600, n_users=20,
                                     n_exact_duplicates=6, n_twin_pairs=10))
pipeline = PreprocessingPipeline()
features = tuple(tuple(pipeline.preprocess(" ".join(d))) for d in corpus_data.documents)
corpus = (tuple(corpus_data.doc_ids), features)

perturbed, edit = add_document(corpus, "new", list(corpus_data.documents[0]))
before = TfidfVectoriser().fit(list(corpus[1]), list(corpus[0]))
after = TfidfVectoriser().fit(list(perturbed[1]), list(perturbed[0]))

shift = analyse_idf_shift(before, after)
print(f"edit: {edit.kind}")
print(f"vocabulary  {shift.n_before} -> {shift.n_after}")
print(f"max |d idf| over the union   {shift.linf:.6e}")
print(f"max |d idf| over shared      {shift.linf_shared:.6e}")
print(f"worst token {shift.worst_token!r} moved {shift.worst_delta:.6e}")
print(f"looseness = linf / linf_shared = {shift.looseness:.3f}")
print("1.0 means the vocabulary was stable and the section 4.2 bound is tight;")
print("larger values mean it is driven by tokens absent on one side (G5).")

## The shift is concentrated in rare terms

Plot `|Δidf|` against `df`. The relationship is the derivative of `ln`, so the
curve should fall as `1/df` -- which is exactly why a single added document can
move a rare term's weight substantially while barely touching a common one.

In [ ]:
import matplotlib.pyplot as plt

points = []
for term_id in range(before.n_features):
    token = before.vocabulary.token_of(term_id)
    after_id = after.vocabulary.id_of(token)
    if after_id is None:
        continue
    df = before.vocabulary.df_of(token)
    delta = abs(after.idf.values[after_id] - before.idf.values[term_id])
    if delta > 0:
        points.append((df, delta))

xs, ys = zip(*points)
figure, axes = plt.subplots(figsize=(6.4, 4))
axes.scatter(xs, ys, s=6, alpha=0.4)
axes.set_xscale("log"); axes.set_yscale("log")
axes.set_xlabel("document frequency before the edit")
axes.set_ylabel(r"$|\Delta \mathrm{idf}|$")
axes.set_title("One added document: IDF shift falls as 1/df")
axes.grid(alpha=0.3)
plt.show()